<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/agent_dark_background.png" align="center" width="20%">
</div>

<br>

# EVALUATING AGENTS USING EDD

<br>

**About:** Build a multi-tool LLM agent with a router and learn to evaluate its behavior using Evaluation-Driven Development (EDD).

**Learning Goals:** (1) Define agent tools (database lookup, data analysis, data visualization) as callable functions. (2) Implement a router that selects and sequences tools in response to a user query. (3) Use the EDD cycle to identify agent failure modes and improve them by rewriting the prompts that drive tool behavior.

**Keywords:** llm agent, tool calling, router, openai, evaluation-driven development, edd

**Prerequisite Knowledge:** (1) Python functions and classes, (2) Basic OpenAI API usage (chat completions), (3) Pandas DataFrames

**Target User:** Python developers or data practitioners who want to understand how to build and iteratively improve a multi-tool LLM agent.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: SETUP](#Part_0)
> #### [PART 1: AGENT TOOLS](#Part_1)
> #### [PART 2: THE ROUTER](#Part_2)
> #### [PART 3: EVALUATION-DRIVEN IMPROVEMENT](#Part_3)

<br>

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP** AND ENVIRONMENT

This notebook requires an OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`, and a local copy of the sales dataset in `data/`. All imports go in the first code cell so that a clean kernel restart produces a runnable notebook from top to bottom.

The agent in this series uses three tools wired together by a router. Part 1 builds the tools. Part 2 builds the router. Part 3 closes the EDD loop by using a better prompt - derived from observing the agent's failure mode - to improve SQL generation quality.

___

**Note:** `gpt-4o-mini` is used for lower cost during development. You can swap `MODEL` to `gpt-4o` for higher-quality outputs at higher cost.

___

In [ ]:
from openai import OpenAI
import pandas as pd
import os
import json
import duckdb
from pydantic import BaseModel, Field
from IPython.display import Markdown
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

MODEL = "gpt-4o-mini"
TRANSACTION_DATA_FILE_PATH = "data/Store_Sales_Price_Elasticity_Promotions_Data.parquet"

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Why should all `import` statements appear in the first code cell rather than scattered throughout the notebook?**

<br>

```python
# Write your explanation here as a comment.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **AGENT TOOLS**: Database, Analysis, Visualization

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/database_lookup_tool_dark.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 1.1: DATABASE LOOKUP TOOL](#Part_1_1)<br>
> [PART 1.2: DATA ANALYSIS TOOL](#Part_1_2)<br>
> [PART 1.3: DATA VISUALIZATION TOOL](#Part_1_3)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: DATABASE LOOKUP TOOL

<br>

The database lookup tool converts a natural-language user request into a SQL query and executes it against a local Parquet file loaded into DuckDB. It runs in three steps:

1. **Prepare the database** - read the Parquet file into an in-memory DuckDB table (created once, reused on subsequent calls via `CREATE TABLE IF NOT EXISTS`).
2. **Generate SQL** - call the LLM with the user's prompt, the table's column names, and the table name. The LLM returns a SQL string.
3. **Execute SQL** - run the returned query against DuckDB and return the result as a formatted string.

This three-step pattern separates natural-language understanding (step 2, LLM) from data access (steps 1 and 3, DuckDB), which makes each step independently testable.

___

**Note:** The `except` block returns an error string rather than raising an exception. This is intentional - the router receives the error as a tool result and can decide how to proceed rather than crashing.

___

In [ ]:
SQL_GENERATION_PROMPT = """
Generate an SQL query based on the prompt that follows. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""

In [ ]:
def generate_sql_query(prompt: str, columns: list[str], table_name: str) -> str:
    """
    Generate a valid SQL query from a natural language prompt.

    Parameters
    ----------
    prompt : str
        Natural language query.
    columns : list[str]
        Column names in the target table.
    table_name : str
        Name of the table to query.

    Returns
    -------
    str
        Generated SQL query string.
    """
    formatted_prompt = SQL_GENERATION_PROMPT.format(
        prompt=prompt,
        columns=columns,
        table_name=table_name
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    return response.choices[0].message.content.strip()


def lookup_sales_data(prompt: str) -> str:
    """
    Query sales data from a Parquet file using LLM-generated SQL.

    Parameters
    ----------
    prompt : str
        Natural language query.

    Returns
    -------
    str
        Query result as formatted text or an error message.
    """
    try:
        table_name = "sales"
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")
        sql_query = generate_sql_query(prompt, df.columns.tolist(), table_name)
        sql_query = sql_query.strip().replace("```sql", "").replace("```", "")
        result = duckdb.sql(sql_query).df()
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {e}" 

In [ ]:
# Test the database lookup tool with a sample query.
# Expected output: a table of rows for store 1320 on 2021-11-01.
prompt = "Show me all the sales for store 1320 on November 1st, 2021"
example_data = lookup_sales_data(prompt)
print(example_data)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The `lookup_sales_data` function uses `CREATE TABLE IF NOT EXISTS` instead of `CREATE TABLE`. Explain why this matters when the tool is called multiple times in the same session.**

<br>

```python
# Write your explanation as a comment, then predict what would happen if you changed it to CREATE TABLE.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: DATA ANALYSIS TOOL

<br>

The data analysis tool takes a string of data (typically the output of the lookup tool) and a question, then asks the LLM to produce an analytical response. It is deliberately simple: one prompt template, one LLM call, one returned string.

The simplicity is a design choice. The tool does not parse or structure the data before passing it to the LLM - it relies on the LLM's ability to read tabular text. This works for moderate-sized results but would fail for very large query outputs. That trade-off is acceptable in this demo context and is worth revisiting in production.

In [ ]:
DATA_ANALYSIS_PROMPT = """
Analyze the following data: {data}
Your job is to answer the following question: {prompt}
"""

In [ ]:
def analyze_sales_data(prompt: str, data: str) -> str:
    """
    Generate an AI-based sales data analysis.

    Args:
        prompt (str): Instruction describing the analysis task.
        data (str): Sales dataset or text input to analyze.

    Returns:
        str: LLM-generated analysis, or a fallback message if empty.
    """
    formatted_prompt = DATA_ANALYSIS_PROMPT.format(data=data, prompt=prompt)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    analysis = response.choices[0].message.content
    return analysis if analysis else "No analysis could be generated" 

In [ ]:
# Test the analysis tool using the data retrieved above.
prompt = "What are some key business insights in the data?"
insights = analyze_sales_data(prompt, example_data)
print(insights)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The `analyze_sales_data` tool passes raw tabular text directly to the LLM without any pre-processing. Write a one-sentence explanation of when this approach breaks down and what you would change.**

<br>

```python
# Write your answer as a comment.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.3: DATA VISUALIZATION TOOL

<br>

The visualization tool generates executable Python code for a chart. It runs in two LLM calls rather than one:

- **Call 1 (`extract_chart_config`)** - given the data and a visualization goal, return a structured configuration object (chart type, axis names, title) using OpenAI's structured output via `response_format=VisualizationConfig`. This call produces a reliable, typed schema.
- **Call 2 (`create_chart`)** - given the configuration, return raw Python code as a string. This call produces executable but unverified code.

Using structured outputs in Call 1 prevents the second call from receiving a malformed configuration. The pattern of "structure first, generate second" is a common reliability technique in LLM pipelines.

In [ ]:
CHART_CONFIGURATION_PROMPT = """
Generate a chart configuration based on this data: {data}
The goal is to show: {visualization_goal}
"""

In [ ]:
class VisualizationConfig(BaseModel):
    """Configuration schema for chart generation."""
    chart_type: str = Field(..., description="Type of chart (e.g., bar, line, scatter).")
    x_axis: str = Field(..., description="Column name for the x-axis.")
    y_axis: str = Field(..., description="Column name for the y-axis.")
    title: str = Field(..., description="Chart title.")

In [ ]:
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """Generate a structured chart configuration from data and a visualization goal."""
    formatted_prompt = CHART_CONFIGURATION_PROMPT.format(
        data=data, visualization_goal=visualization_goal
    )
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
        response_format=VisualizationConfig,
    )
    try:
        content = response.choices[0].message.parsed
        return {
            "chart_type": content.chart_type,
            "x_axis": content.x_axis,
            "y_axis": content.y_axis,
            "title": content.title,
            "data": data
        }
    except Exception:
        return {
            "chart_type": "bar",
            "x_axis": "date",
            "y_axis": "value",
            "title": visualization_goal,
            "data": data
        }


CREATE_CHART_PROMPT = """
Write Python code to create a chart based on the configuration below.
Return only the code, no other text.
config: {config}
"""


def create_chart(config: dict) -> str:
    """Generate Python code for a chart from a configuration dict."""
    formatted_prompt = CREATE_CHART_PROMPT.format(config=config)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    code = response.choices[0].message.content
    code = code.replace("```python", "").replace("```", "").strip()
    return code


def generate_visualization(data: str, visualization_goal: str) -> str:
    """Generate Python code for a visualization from data and a goal description."""
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code

In [ ]:
# Test the visualization tool.
visualization_goal = "A bar chart of sales by SKU. Put SKU_Coded on the x-axis and Total_Sale_Value on the y-axis."
viz_code = generate_visualization(example_data, visualization_goal)
print(viz_code)

In [ ]:
# Run the generated code.
exec(viz_code)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The visualization tool uses two LLM calls: one with structured output and one that returns free-form code. Write the `generate_visualization` function call sequence in pseudocode, and label which step benefits from structured outputs and why.**

<br>

```python
# Write your pseudocode and explanation as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **THE ROUTER**: Tool Selection and Orchestration

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/router.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 2.1: TOOL SCHEMA](#Part_2_1)<br>
> [PART 2.2: ROUTER LOOP](#Part_2_2)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: TOOL SCHEMA

<br>

The router is an LLM with access to a list of tools described in the OpenAI function-calling schema. Each tool schema has three required parts:

- **name** - matches the Python function name exactly. The router uses this string to dispatch to the right function.
- **description** - natural language text the LLM reads to decide when to call this tool. Poorly written descriptions lead to wrong tool selection.
- **parameters** - JSON Schema describing the function's arguments. The LLM fills these in from the user's query.

The `tool_implementations` dict maps tool names (strings) to the actual Python functions, so the dispatch loop can call the right function from the name the LLM returned.

<strong style="color:red">KEY CONSIDERATION:</strong> The tool description is the router's decision signal. A description like "look up data" is ambiguous; "Look up sales transaction data from the Store Sales Price Elasticity dataset" is not. Write descriptions as if you are explaining to a non-technical colleague when to use the tool.

In [ ]:
# Tool schemas define the interface the LLM router uses to select and call tools.
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "Look up sales transaction data from the Store Sales Price Elasticity Promotions dataset",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data",
            "description": "Analyze sales data to extract business insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool output."},
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["data", "prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "Generate executable Python code to create a data visualization",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool output."},
                    "visualization_goal": {"type": "string", "description": "Description of the chart to create."}
                },
                "required": ["data", "visualization_goal"]
            }
        }
    }
]

# Maps tool names returned by the LLM to the actual Python functions.
tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data,
    "generate_visualization": generate_visualization
}

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **If the `analyze_sales_data` description were changed to "Do something with data", what failure mode would you expect from the router? Write a short test question that would expose this failure.**

<br>

```python
# Write your failure-mode prediction and test question as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: ROUTER LOOP

<br>

The router runs as a `while True` loop. Each iteration:

1. Calls the LLM with the current message history and the tool list.
2. Appends the LLM's response to the message history (this preserves context across turns).
3. Checks whether the LLM requested any tool calls.
4. If yes: calls each tool, appends each result to the message history as a `tool` role message, then loops again.
5. If no: the LLM has produced a final answer - return it.

This loop terminates when the LLM stops requesting tools. The LLM decides this based on whether it believes it has enough information to answer the user's query.

___

**Note:** The system prompt is inserted at the start of the message list only if it is not already present. This avoids duplicating the system prompt if `run_agent` is called multiple times with the same list.

___

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.
"""


def handle_tool_calls(tool_calls, messages):
    """Execute each tool the LLM requested and append results to the message list."""
    for tool_call in tool_calls:
        function = tool_implementations[tool_call.function.name]
        function_args = json.loads(tool_call.function.arguments)
        result = function(**function_args)
        messages.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return messages


def run_agent(system_prompt, messages):
    """Run the router loop until the LLM produces a final answer."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    if not any(isinstance(m, dict) and m.get("role") == "system" for m in messages):
        messages.insert(0, {"role": "system", "content": system_prompt})

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        messages.append(response.choices[0].message)
        tool_calls = response.choices[0].message.tool_calls

        if tool_calls:
            messages = handle_tool_calls(tool_calls, messages)
        else:
            return response.choices[0].message.content

In [ ]:
# Run the agent with a multi-step query that requires data lookup, analysis, and visualization.
user_question = "Show me the code for a scatterplot of sales by store in November 2021, and tell me what trends you see."
results = run_agent(SYSTEM_PROMPT, user_question)
Markdown(results)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The router loop appends the LLM's response object to `messages` before calling any tools. Why is this ordering important for the LLM's context on the next iteration?**

<br>

```python
# Write your explanation as a comment.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **EVALUATION-DRIVEN IMPROVEMENT**: Closing the Loop

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/spans_agent.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

EDD treats agent improvement as a data problem. Instead of guessing what to fix, you collect traces, identify failure patterns from the data, then make a targeted change.

In this agent, the database lookup tool is the most likely point of failure. The original `SQL_GENERATION_PROMPT` gives the LLM only three inputs: the user's prompt, the column names, and the table name. It imposes no formatting constraints and provides no SQL style guidance. This works on simple queries but degrades on:

- Ambiguous filter conditions (e.g., "top stores" without a defined ranking criterion)
- Queries that require aggregation across date ranges
- Queries where the LLM hallucinates a column name not in the schema

The improved prompt below adds schema context, step-by-step reasoning instructions, output format constraints, and an explicit fallback for unsupported queries. Each of these additions directly addresses a known failure mode.

___

**Note:** Prompt improvement in EDD is not guessing - it is a response to observed failures. In the next notebook (`02_tracing_agents.ipynb`), you will add OpenTelemetry tracing so that failures become visible as structured spans rather than requiring manual inspection of print outputs.

___

In [ ]:
# Original prompt (baseline)
SQL_GENERATION_PROMPT_ORIGINAL = """
Generate an SQL query based on the prompt that follows. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""

# Improved prompt - addresses ambiguous queries, column hallucination, and output format.
SQL_GENERATION_PROMPT_IMPROVED = """
    You are an expert SQL developer. Your task is to generate a syntactically valid SQL query
    that answers the user's request based on the provided table schema.
    You must return **only** the SQL code, formatted cleanly with uppercase SQL keywords.

    Follow these steps internally before producing the final answer:
    1. Analyze the user's intent.
    2. Identify which columns and filters are relevant.
    3. Construct a valid SQL SELECT statement that answers the query.

    Schema Information:
    - Table Name: {table_name}
    - Available Columns: {columns}

    User Request:
    "{prompt}"

    Guidelines:
    - Return only the SQL query (no explanations, comments, or markdown).
    - Use exact column names as provided - do not invent or rename any.
    - Prefer simple, interpretable SQL (avoid unnecessary nesting or joins).
    - If the user request is ambiguous or impossible, return:
      SELECT 'ERROR: Ambiguous or Unsupported Query' AS message;
""" 

In [ ]:
# Compare the two prompts on a query that tests column constraint adherence.
# The improved prompt should return an error for a query referencing a non-existent column.

def generate_sql_query_v2(prompt: str, columns: list[str], table_name: str) -> str:
    """SQL generation using the improved prompt."""
    formatted_prompt = SQL_GENERATION_PROMPT_IMPROVED.format(
        prompt=prompt, columns=columns, table_name=table_name
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    return response.choices[0].message.content.strip()


# Test both prompts on the same ambiguous query.
df_schema = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
columns = df_schema.columns.tolist()
table_name = "sales"

test_query = "Show me all sales where the discount percentage is above 20%"

print("=== Original prompt output ===")
print(generate_sql_query(test_query, columns, table_name))

print()
print("=== Improved prompt output ===")
print(generate_sql_query_v2(test_query, columns, table_name))

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The improved SQL prompt adds "do not invent or rename any columns". Write a test query that specifically tests whether this constraint is working, and run it against both versions of `generate_sql_query`. Record what each version returns.**

<br>

```python
# Write your test query and run it against generate_sql_query and generate_sql_query_v2.
test_query = "..."
print(generate_sql_query(test_query, columns, table_name))
print(generate_sql_query_v2(test_query, columns, table_name))
```

<hr style="border: 2px solid#003262;" />

___

**Next:** `02_tracing_agents.ipynb` adds OpenTelemetry instrumentation to this same agent using Arize Phoenix. With tracing in place, every LLM call and tool invocation becomes a structured span - making failure modes visible without manual print debugging.

___

<hr style="border: 6px solid#003262;" />